# Bước 09: Kiểm chứng trễ pha trên tập test - vẽ từng site
Dự án: Tốt nghiệp - Energy Forecasting - Nhóm thực hiện: The Outliers

## 1. TỔNG QUAN VÀ MỤC TIÊU

Notebook này **không train gì cả**, chỉ đọc lại kết quả dự báo đã lưu ở bước 07 và trả lời 3 câu hỏi:

1. Mô hình có bị **trễ pha** không - ở đâu, ngày nào cụ thể?
2. Trễ ở **site nào**, **thời điểm nào** - có phải chỗ nào cũng trễ như nhau không?
3. Trễ nặng vào lúc nào trong ngày - lúc trời chuyển mây hay lúc bình minh/hoàng hôn?

### Cách kiểm chứng - LOCAL, không dùng RMSE/tương quan tổng hợp
RMSE hay tương quan tính gộp trên cả năm dữ liệu sẽ bị trung bình hoá và có thể che mất bug
chỉ xảy ra ở 1 site/1 ngày cụ thể - một site lệch nặng có thể bị 41 site còn lại "kéo" số liệu
tổng hợp về trông như bình thường. Vì vậy notebook này đo LOCAL: với MỖI (site, ngày), tìm giờ
đỉnh sản lượng thực tế và giờ đỉnh dự báo (trong khung ban ngày), rồi so lệch bằng phút:

```
lech_phut = gio_dinh_du_bao - gio_dinh_thuc_te
```

Dương = dự báo đến SAU thực tế (trễ/dịch phải). Kết quả là 1 bảng đầy đủ 42 site x mọi ngày trong
tập test - 1 ô lệch bất thường trong bảng đó là 1 bug thật, không được bỏ qua chỉ vì trung vị toàn
bộ trông đẹp.

### Nguồn dữ liệu
`../../data/model/v4/07_final_test/prediction_audit.parquet` - kết quả của notebook 07 trên tập test.


## 2. Import thư viện và khai báo tham số


In [13]:
# ── Gioi han thread: may i5-12450HX co 8 core / 12 thread (4 P-core + 4 E-core).
# Dung het 12 thread lam cac thread tranh nhau va CHAM HON. Dat 6 de bam P-core,
# con lai de cho Jupyter va he dieu hanh. Phai set TRUOC khi numpy nap moi co tac dung.
import os

for _bien in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
              'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ.setdefault(_bien, '6')

import os
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go

warnings.filterwarnings('ignore')

SITE_COL = 'site_id'
TIMESTAMP_COL = 'timestamp'

AUDIT_PATH = '../../data/model/v4/07_final_test/prediction_audit.parquet'
OUTPUT_DIR = '../../data/model/v4/09_kiem_chung_tre_pha'
os.makedirs(OUTPUT_DIR, exist_ok=True)

HORIZON_XEM = 1              # xem h1 truoc, doi thanh 4 de xem h4
DICH_TU, DICH_DEN = -4, 4    # quet do dich tu -4 den +4 buoc (moi buoc 15 phut)
SO_NGAY_VE = 14              # so ngay ve trong bieu do co dropdown chon site

print("Đã import thư viện và khai báo tham số.")
print(f"- File kết quả  : {AUDIT_PATH}")
print(f"- Horizon xem   : h{HORIZON_XEM}")
print(f"- Quét độ dịch  : {DICH_TU} đến {DICH_DEN} bước (1 bước = 15 phút)")
print(f"- Thư mục xuất  : {OUTPUT_DIR}")


Đã import thư viện và khai báo tham số.
- File kết quả  : ../../data/model/v4/07_final_test/prediction_audit.parquet
- Horizon xem   : h1
- Quét độ dịch  : -4 đến 4 bước (1 bước = 15 phút)
- Thư mục xuất  : ../../data/model/v4/09_kiem_chung_tre_pha


## 3. Đọc kết quả dự báo của tập test


In [14]:
if not os.path.exists(AUDIT_PATH):
    raise FileNotFoundError(f"Chưa có {AUDIT_PATH}. Chạy noteb07_final_test trước.")

au = pd.read_parquet(AUDIT_PATH)
yt_col, yp_col = f'y_true_h{HORIZON_XEM}', f'y_pred_h{HORIZON_XEM}'

df = au[[SITE_COL, TIMESTAMP_COL, 'energy_source', 'is_daylight', yt_col, yp_col]].copy()
df = df.dropna(subset=[yt_col, yp_col])
df = df.sort_values([SITE_COL, TIMESTAMP_COL]).reset_index(drop=True)
df = df.rename(columns={yt_col: 'thuc_te', yp_col: 'du_bao'})

print(f"Tổng số dòng    : {len(df):,}")
print(f"Số site         : {df[SITE_COL].nunique()}")
print(f"Khoảng thời gian: {df[TIMESTAMP_COL].min()} -> {df[TIMESTAMP_COL].max()}")
print(f"Tỉ lệ ban ngày  : {df['is_daylight'].mean() * 100:.2f}%")
display(df.head(3))


Tổng số dòng    : 475,599
Số site         : 40
Khoảng thời gian: 2021-12-18 09:30:00 -> 2022-04-23 23:30:00
Tỉ lệ ban ngày  : 53.70%


,site_id,timestamp,energy_source,is_daylight,thuc_te,du_bao
0,1,2021-12-18 09:30:00,measured,True,5.6250,10.669604
1,1,2021-12-18 09:45:00,measured,True,6.4375,10.831930
2,1,2021-12-18 10:00:00,measured,True,9.1875,8.624905


## 4. Gộp nhãn outlier vào kết quả dự báo

`prediction_audit.parquet` không mang cột `outlier_group`, phải lấy lại từ `05_selected`.

Điểm cần soi: quy trình hiện tại **vẫn đưa outlier vào train** - `exclude_from_training` chỉ chặn
dòng target bịa, không chặn `outlier_group != normal`. Hệ quả là mô hình học luôn cả các phốc bất thường.


In [15]:
TEST_SELECTED = '../../data/model/v4/05_selected/v4_test_selected.parquet'

_ol = pd.read_parquet(TEST_SELECTED, columns=[SITE_COL, TIMESTAMP_COL, 'outlier_group', 'exclude_from_training'])
df = df.merge(_ol, on=[SITE_COL, TIMESTAMP_COL], how='left')
df['la_outlier'] = df['outlier_group'].notna() & (df['outlier_group'] != 'normal')

print("--- PHAN BO NHAN OUTLIER TREN TAP TEST ---")
print(df['outlier_group'].value_counts(dropna=False).to_string())
print()
print(f"Tong dong outlier      : {int(df['la_outlier'].sum()):,} / {len(df):,} "
      f"({df['la_outlier'].mean() * 100:.2f}%)")

_bi_chan = df.loc[df['la_outlier'], 'exclude_from_training'].fillna(False).astype(bool)
print(f"Trong so outlier do, bi exclude_from_training = True: {int(_bi_chan.sum()):,} "
      f"({_bi_chan.mean() * 100:.2f}%)")
print("Neu con so tren gan 0 thi dung la outlier duoc dua thang vao train.")
del _ol
display(df.head(3))


--- PHAN BO NHAN OUTLIER TREN TAP TEST ---
outlier_group
normal                 474300
gmm_if_consensus          899
other_physical_rule       242
multiple_rules            158

Tong dong outlier      : 1,299 / 475,599 (0.27%)
Trong so outlier do, bi exclude_from_training = True: 0 (0.00%)
Neu con so tren gan 0 thi dung la outlier duoc dua thang vao train.


,site_id,timestamp,energy_source,is_daylight,thuc_te,du_bao,outlier_group,exclude_from_training,la_outlier
0,1,2021-12-18 09:30:00,measured,True,5.6250,10.669604,normal,False,False
1,1,2021-12-18 09:45:00,measured,True,6.4375,10.831930,normal,False,False
2,1,2021-12-18 10:00:00,measured,True,9.1875,8.624905,normal,False,False


## 5. Lệch đỉnh LOCAL từng site x từng ngày (không dùng RMSE)

Không tính RMSE/tương quan tổng hợp trên cả năm nữa — số liệu đó bị trung bình hoá,
che mất bug chỉ xảy ra ở 1 site/1 ngày cụ thể. Thay vào đó: với MỖI (site, ngày),
tìm giờ đỉnh sản lượng thực tế và giờ đỉnh dự báo (trong khung ban ngày), rồi so lệch
bằng phút. Đây là phép đo cục bộ thật sự — 1 ô lệch trong bảng dưới đây là 1 bug thật,
không bị pha loãng bởi các site/ngày khác.

In [16]:
def lech_dinh_moi_ngay(d):
    """Voi MOI (site, ngay), tim gio dinh cua thuc te va cua du bao (chi ban ngay),
    tra ve do lech THOI DIEM (phut). Day la phep do LOCAL, khong gop chung nhieu ngay/site
    lai voi nhau nen khong the bi trung binh hoa che mat bug."""
    w = d[d['is_daylight'].fillna(False).astype(bool)].copy()
    w['ngay'] = w[TIMESTAMP_COL].dt.date
    rows = []
    for (s, ngay), g in w.groupby([SITE_COL, 'ngay']):
        if len(g) < 4 or g['thuc_te'].max() <= 0:
            continue
        t_dinh_thuc = g.loc[g['thuc_te'].idxmax(), TIMESTAMP_COL]
        t_dinh_du_bao = g.loc[g['du_bao'].idxmax(), TIMESTAMP_COL]
        lech_phut = (t_dinh_du_bao - t_dinh_thuc).total_seconds() / 60.0
        rows.append({
            'site_id': s, 'ngay': ngay,
            'gio_dinh_thuc_te': t_dinh_thuc.strftime('%H:%M'),
            'gio_dinh_du_bao': t_dinh_du_bao.strftime('%H:%M'),
            'phut_trong_gio': int(t_dinh_thuc.minute),  # vi tri trong gio nguon (0-59)
            'lech_phut': lech_phut,   # duong = du bao den SAU thuc te (tre / dich phai)
            'thuc_te_dinh': float(g['thuc_te'].max()),
            'du_bao_dinh': float(g['du_bao'].max()),
        })
    return pd.DataFrame(rows)


df_lech = lech_dinh_moi_ngay(df)
df_lech.to_csv(f'{OUTPUT_DIR}/lech_dinh_moi_ngay_h{HORIZON_XEM}.csv', index=False)

print(f"--- LECH DINH LOCAL: {df_lech['site_id'].nunique()} site x {len(df_lech):,} ngay ---")
print("lech_phut duong = du bao den SAU thuc te (dich phai / tre). am = du bao den SOM.")
print()
print(df_lech['lech_phut'].describe().round(2).to_string())
print()
so_dich_phai = int((df_lech['lech_phut'] > 0).sum())
so_dung = int((df_lech['lech_phut'] == 0).sum())
so_dich_trai = int((df_lech['lech_phut'] < 0).sum())
print(f"So ngay du bao dich PHAI (tre)  : {so_dich_phai:,} / {len(df_lech):,} ({so_dich_phai/len(df_lech)*100:.1f}%)")
print(f"So ngay dung khop (lech = 0)    : {so_dung:,} / {len(df_lech):,} ({so_dung/len(df_lech)*100:.1f}%)")
print(f"So ngay du bao dich TRAI (som)  : {so_dich_trai:,} / {len(df_lech):,} ({so_dich_trai/len(df_lech)*100:.1f}%)")
print()
print("--- 15 CA LECH NANG NHAT (theo tri tuyet doi lech_phut) ---")
display(df_lech.reindex(df_lech['lech_phut'].abs().sort_values(ascending=False).index).head(15))
print(f"Đã lưu {OUTPUT_DIR}/lech_dinh_moi_ngay_h{HORIZON_XEM}.csv")

print()
print("--- KIEM TRA: lech_phut co lien quan toi VI TRI TRONG GIO khong (khong dung RMSE) ---")
print("Neu do gioi han thoi tiet hourly: dinh thuc te xay ra cang GAN CUOI gio (phut_trong_gio cao)")
print("thi lech_phut cang LON, vi model chua kip nhan thong tin gio moi.")
print("Neu KHONG thay xu huong nay -> khong phai do du lieu thoi tiet, van la bug code can tim tiep.")
_theo_vi_tri = df_lech.groupby(pd.cut(df_lech['phut_trong_gio'], bins=[-1, 14, 29, 44, 59],
                                       labels=['0-14', '15-29', '30-44', '45-59']),
                                observed=True)['lech_phut'].agg(so_ngay='count', lech_trung_vi='median').reset_index()
_theo_vi_tri.columns = ['phut_trong_gio_nhom', 'so_ngay', 'lech_trung_vi_phut']
display(_theo_vi_tri)

fig_vt = go.Figure()
fig_vt.add_trace(go.Scatter(x=df_lech['phut_trong_gio'], y=df_lech['lech_phut'], mode='markers',
                            marker=dict(size=5, opacity=0.35, color='#1f77b4'), name='moi (site, ngay)'))
fig_vt.update_layout(title='Lech dinh (phut) theo vi tri trong gio dinh thuc te xay ra',
                     xaxis_title='Phut trong gio (0-59)', yaxis_title='Lech dinh (phut, + = tre)',
                     height=420, template='plotly_white')
fig_vt.show()
_theo_vi_tri.to_csv(f'{OUTPUT_DIR}/lech_theo_vi_tri_trong_gio_h{HORIZON_XEM}.csv', index=False)
print(f"Đã lưu {OUTPUT_DIR}/lech_theo_vi_tri_trong_gio_h{HORIZON_XEM}.csv")


--- LECH DINH LOCAL: 40 site x 5,080 ngay ---
lech_phut duong = du bao den SAU thuc te (dich phai / tre). am = du bao den SOM.

count    5080.00
mean       -1.05
std        78.63
min      -495.00
25%       -45.00
50%        15.00
75%        45.00
max       270.00

So ngay du bao dich PHAI (tre)  : 2,791 / 5,080 (54.9%)
So ngay dung khop (lech = 0)    : 384 / 5,080 (7.6%)
So ngay du bao dich TRAI (som)  : 1,905 / 5,080 (37.5%)

--- 15 CA LECH NANG NHAT (theo tri tuyet doi lech_phut) ---


,site_id,ngay,gio_dinh_thuc_te,gio_dinh_du_bao,phut_trong_gio,lech_phut,thuc_te_dinh,du_bao_dinh
128,2,2021-12-19,17:30,09:15,30,-495.0,10.468750,7.020079
382,4,2021-12-19,17:30,09:30,30,-480.0,1.398438,0.657583
1,1,2021-12-19,17:30,09:45,30,-465.0,13.687500,8.223557
509,5,2021-12-19,17:30,09:45,30,-465.0,1.109375,0.618929
255,3,2021-12-19,17:30,09:45,30,-465.0,8.375000,4.802207
4954,42,2021-12-19,15:15,09:30,15,-345.0,5.562500,3.253437
1982,16,2022-03-05,17:45,12:15,45,-330.0,1.523438,1.307225
1823,15,2022-02-01,15:45,10:30,45,-315.0,16.609375,13.948430
614,5,2022-04-03,15:00,10:00,0,-300.0,0.601562,0.543950
1612,13,2022-03-16,16:45,11:45,45,-300.0,2.687500,2.213286


Đã lưu ../../data/model/v4/09_kiem_chung_tre_pha/lech_dinh_moi_ngay_h1.csv

--- KIEM TRA: lech_phut co lien quan toi VI TRI TRONG GIO khong (khong dung RMSE) ---
Neu do gioi han thoi tiet hourly: dinh thuc te xay ra cang GAN CUOI gio (phut_trong_gio cao)
thi lech_phut cang LON, vi model chua kip nhan thong tin gio moi.
Neu KHONG thay xu huong nay -> khong phai do du lieu thoi tiet, van la bug code can tim tiep.


,phut_trong_gio_nhom,so_ngay,lech_trung_vi_phut
0,0-14,1052,15.0
1,15-29,1112,15.0
2,30-44,1631,15.0
3,45-59,1285,15.0


Đã lưu ../../data/model/v4/09_kiem_chung_tre_pha/lech_theo_vi_tri_trong_gio_h1.csv


## 6. Tổng hợp lệch đỉnh theo TỪNG site (từ bảng local ở trên)

Chỉ tổng hợp lại bảng `df_lech` để xếp hạng site — vẫn dựa 100% trên số liệu local
từng ngày, không tính lại RMSE nào.

In [17]:
df_site = df_lech.groupby(SITE_COL)['lech_phut'].agg(
    so_ngay='count',
    lech_trung_vi='median',
    lech_trung_binh='mean',
    so_ngay_dich_phai=lambda x: int((x > 0).sum()),
    ca_lech_nang_nhat='max',
).reset_index()
df_site['ty_le_dich_phai_%'] = (df_site['so_ngay_dich_phai'] / df_site['so_ngay'] * 100).round(1)
df_site = df_site.sort_values('lech_trung_vi', ascending=False).reset_index(drop=True)

print("--- LECH DINH THEO TUNG SITE (sap xep site tre nang nhat len dau) ---")
print("lech_trung_vi duong lon = site do thuong xuyen du bao den SAU thuc te (tre).")
display(df_site.round(2))

print(f"\nSo site co lech_trung_vi > 0 (thien ve tre): {(df_site['lech_trung_vi'] > 0).sum()}/{len(df_site)}")
df_site.to_csv(f'{OUTPUT_DIR}/lech_dinh_theo_site_h{HORIZON_XEM}.csv', index=False)
print(f"Đã lưu {OUTPUT_DIR}/lech_dinh_theo_site_h{HORIZON_XEM}.csv")


--- LECH DINH THEO TUNG SITE (sap xep site tre nang nhat len dau) ---
lech_trung_vi duong lon = site do thuong xuyen du bao den SAU thuc te (tre).


,site_id,so_ngay,lech_trung_vi,lech_trung_binh,so_ngay_dich_phai,ca_lech_nang_nhat,ty_le_dich_phai_%
0,4,127,30.0,0.35,75,210.0,59.1
1,5,127,30.0,14.06,91,255.0,71.7
2,38,127,30.0,7.56,75,210.0,59.1
3,23,127,30.0,-2.95,74,240.0,58.3
4,26,127,30.0,1.89,74,210.0,58.3
5,14,127,30.0,6.73,73,210.0,57.5
6,16,127,30.0,3.66,73,180.0,57.5
7,15,127,30.0,1.18,69,255.0,54.3
8,31,127,30.0,2.24,76,225.0,59.8
9,30,127,30.0,5.20,76,225.0,59.8



So site co lech_trung_vi > 0 (thien ve tre): 32/40
Đã lưu ../../data/model/v4/09_kiem_chung_tre_pha/lech_dinh_theo_site_h1.csv


## 7. Biểu đồ chọn site (nhẹ, không nhúng hết dữ liệu vào file)

Trước đây nhúng cả 42 site × toàn bộ tập test vào 1 figure Plotly làm file phình lên
49,5 MB, VS Code không mở nổi. Đổi sang `ipywidgets.interact`: mỗi lần đổi site chỉ vẽ
lại 1 site, notebook chỉ lưu lại ảnh của lần xem cuối cùng - vẫn phủ đủ toàn bộ khoảng
test cho MỌI site khi anh tự chọn qua dropdown, không cắt bớt ngày nào.

In [18]:
from ipywidgets import interact, Dropdown

sites = sorted(df[SITE_COL].unique().tolist())
_t_start_all = df[TIMESTAMP_COL].min()
_t_end_all = df[TIMESTAMP_COL].max()


def _ve_site(site_id):
    d = df[(df[SITE_COL] == site_id) & (df[TIMESTAMP_COL].between(_t_start_all, _t_end_all))].sort_values(TIMESTAMP_COL)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=d[TIMESTAMP_COL], y=d['thuc_te'], name='Thực tế',
                             mode='lines', line=dict(width=2.0, color='#1f77b4')))
    fig.add_trace(go.Scatter(x=d[TIMESTAMP_COL], y=d['du_bao'], name='Dự báo',
                             mode='lines', line=dict(width=1.6, color='#d62728')))
    fig.add_trace(go.Scatter(x=d[TIMESTAMP_COL], y=d['thuc_te'].shift(1), name='Thực tế dịch lùi 1 bước',
                             mode='lines', line=dict(width=1.2, dash='dot', color='#7f7f7f')))
    _o = d[d['la_outlier']]
    fig.add_trace(go.Scatter(x=_o[TIMESTAMP_COL], y=_o['thuc_te'], name='Điểm outlier',
                             mode='markers', marker=dict(size=8, symbol='x', color='#ff7f0e')))
    fig.update_layout(
        title=f'Site {site_id} - h{HORIZON_XEM} - toàn bộ tập test ({_t_start_all.date()} -> {_t_end_all.date()})',
        xaxis_title='Thời gian', yaxis_title='Sản lượng (kWh)',
        hovermode='x unified', height=520, template='plotly_white',
    )
    fig.show()


interact(_ve_site, site_id=Dropdown(options=sites, value=sites[0], description='Site:'))
print(f"Sẵn sàng {len(sites)} site. Đổi dropdown 'Site:' phía trên để xem từng site.")
print("Dau X mau cam la diem bi gan nhan outlier - xem duong do co bam theo chung khong.")

interactive(children=(Dropdown(description='Site:', options=(1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15…

Sẵn sàng 40 site. Đổi dropdown 'Site:' phía trên để xem từng site.
Dau X mau cam la diem bi gan nhan outlier - xem duong do co bam theo chung khong.


## 8. Zoom vào ngày lệch đỉnh nặng nhất của site trễ nhất

Lấy đúng site + ngày có `lech_phut` lớn nhất từ bảng local `df_lech` (không phải từ RMSE).

In [19]:
_ca_te_nhat = df_lech.reindex(df_lech['lech_phut'].abs().sort_values(ascending=False).index).iloc[0]
SITE_TE_NHAT = _ca_te_nhat['site_id']
ngay_te = _ca_te_nhat['ngay']

d = df[(df[SITE_COL] == SITE_TE_NHAT) & (df[TIMESTAMP_COL].dt.date == ngay_te)].sort_values(TIMESTAMP_COL).copy()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=d[TIMESTAMP_COL], y=d['thuc_te'], name='Thực tế',
                          mode='lines+markers', line=dict(width=2.6, color='#1f77b4')))
fig2.add_trace(go.Scatter(x=d[TIMESTAMP_COL], y=d['du_bao'], name='Dự báo',
                          mode='lines+markers', line=dict(width=2.1, color='#d62728')))
fig2.add_vline(x=pd.Timestamp(f"{ngay_te} {_ca_te_nhat['gio_dinh_thuc_te']}"), line_dash='dot',
              line_color='#1f77b4', annotation_text='dinh thuc te')
fig2.add_vline(x=pd.Timestamp(f"{ngay_te} {_ca_te_nhat['gio_dinh_du_bao']}"), line_dash='dot',
              line_color='#d62728', annotation_text='dinh du bao')
fig2.update_layout(
    title=f'Site {SITE_TE_NHAT} - ngay lech dinh nang nhat {ngay_te} - h{HORIZON_XEM}',
    xaxis_title='Thời gian', yaxis_title='Sản lượng (kWh)',
    hovermode='x unified', height=520, template='plotly_white',
)
fig2.show()

print(f"Site lệch nặng nhất : {SITE_TE_NHAT}")
print(f"Ngày lệch nặng nhất : {ngay_te}")
print(f"Đỉnh thực tế lúc    : {_ca_te_nhat['gio_dinh_thuc_te']} (giá trị {_ca_te_nhat['thuc_te_dinh']:.3f})")
print(f"Đỉnh dự báo lúc     : {_ca_te_nhat['gio_dinh_du_bao']} (giá trị {_ca_te_nhat['du_bao_dinh']:.3f})")
print(f"Lệch                : {_ca_te_nhat['lech_phut']:+.0f} phút "
      f"({'du bao den SAU (tre)' if _ca_te_nhat['lech_phut'] > 0 else 'du bao den SOM' if _ca_te_nhat['lech_phut'] < 0 else 'khop'})")


Site lệch nặng nhất : 2
Ngày lệch nặng nhất : 2021-12-19
Đỉnh thực tế lúc    : 17:30 (giá trị 10.469)
Đỉnh dự báo lúc     : 09:15 (giá trị 7.020)
Lệch                : -495 phút (du bao den SOM)


## 9. Lệch đỉnh nặng vào khung giờ nào trong ngày

Gộp `df_lech` (đã tính local từng ngày) theo giờ đỉnh thực tế xảy ra — không tính lại
RMSE gì thêm, chỉ tổng hợp lại con số local đã có.

In [20]:
w = df_lech.copy()
w['gio_dinh'] = pd.to_datetime(w['gio_dinh_thuc_te'], format='%H:%M').dt.hour

theo_gio = w.groupby('gio_dinh')['lech_phut'].agg(
    so_ngay='count', lech_trung_vi='median', lech_trung_binh='mean',
    so_ngay_dich_phai=lambda x: int((x > 0).sum()),
).reset_index()
theo_gio['ty_le_dich_phai_%'] = (theo_gio['so_ngay_dich_phai'] / theo_gio['so_ngay'] * 100).round(1)

print("--- LECH DINH THEO KHUNG GIO DINH XAY RA (tu bang local, khong RMSE) ---")
display(theo_gio.round(2))

fig3 = go.Figure()
fig3.add_trace(go.Bar(x=theo_gio['gio_dinh'], y=theo_gio['lech_trung_vi'],
                      marker_color=['#d62728' if v > 0 else '#2ca02c' for v in theo_gio['lech_trung_vi']],
                      name='Lệch đỉnh trung vị (phút)'))
fig3.update_layout(title=f'Lệch đỉnh trung vị theo giờ đỉnh xảy ra - h{HORIZON_XEM}',
                   xaxis_title='Giờ đỉnh thực tế', yaxis_title='Lệch đỉnh trung vị (phút, + = trễ)',
                   height=420, template='plotly_white')
fig3.show()

_gio_te = int(theo_gio.loc[theo_gio['lech_trung_vi'].abs().idxmax(), 'gio_dinh'])
print(f"Giờ có lệch đỉnh trung vị lớn nhất: {_gio_te} giờ")
theo_gio.to_csv(f'{OUTPUT_DIR}/lech_dinh_theo_gio_h{HORIZON_XEM}.csv', index=False)
print(f"Đã lưu {OUTPUT_DIR}/lech_dinh_theo_gio_h{HORIZON_XEM}.csv")


--- LECH DINH THEO KHUNG GIO DINH XAY RA (tu bang local, khong RMSE) ---


,gio_dinh,so_ngay,lech_trung_vi,lech_trung_binh,so_ngay_dich_phai,ty_le_dich_phai_%
0,7,2,37.5,37.50,2,100.0
1,8,5,60.0,57.00,5,100.0
2,9,21,60.0,91.43,20,95.2
3,10,133,60.0,87.97,123,92.5
4,11,594,45.0,46.94,505,85.0
5,12,1233,30.0,30.94,882,71.5
6,13,1608,0.0,-2.27,782,48.6
7,14,952,-45.0,-40.73,300,31.5
8,15,376,-60.0,-67.82,123,32.7
9,16,141,-105.0,-106.06,42,29.8


Giờ có lệch đỉnh trung vị lớn nhất: 16 giờ
Đã lưu ../../data/model/v4/09_kiem_chung_tre_pha/lech_dinh_theo_gio_h1.csv


## 10. Mô hình có dự báo trùng cả phốc outlier không

Nếu mô hình bám sát điểm outlier thay vì bỏ qua nó, nghĩa là outlier đã được học như dữ liệu thật.
So sai số **tại điểm outlier** với sai số **tại điểm bình thường**: nếu sai số tại outlier không lớn
hơn bao nhiêu, tức là mô hình đang dự báo trúng cả phốc.


In [21]:
def _mae(x):
    return float(np.mean(np.abs(x['du_bao'] - x['thuc_te'])))

bang_ol = []
for s, d in df.groupby(SITE_COL):
    d_ol = d[d['la_outlier']]
    d_nm = d[~d['la_outlier']]
    if len(d_ol) == 0:
        continue
    bang_ol.append({
        'site_id': s,
        'so_outlier': len(d_ol),
        'ty_le_outlier_%': round(len(d_ol) / len(d) * 100, 3),
        'mae_tai_outlier': round(_mae(d_ol), 4),
        'mae_tai_binh_thuong': round(_mae(d_nm), 4) if len(d_nm) else np.nan,
        'tuong_quan_tai_outlier': round(float(np.corrcoef(d_ol['du_bao'], d_ol['thuc_te'])[0, 1]), 5)
                                  if len(d_ol) > 2 and d_ol['thuc_te'].std() > 0 else np.nan,
        'thuc_te_max_outlier': round(float(d_ol['thuc_te'].max()), 3),
        'du_bao_max_outlier': round(float(d_ol['du_bao'].max()), 3),
    })

df_ol = pd.DataFrame(bang_ol).sort_values('so_outlier', ascending=False).reset_index(drop=True)
print("--- SAI SO TAI DIEM OUTLIER SO VOI DIEM BINH THUONG, THEO SITE ---")
print("tuong_quan_tai_outlier cao nghia la mo hinh bam theo ca phoc outlier.")
display(df_ol)
df_ol.to_csv(f'{OUTPUT_DIR}/outlier_theo_site_h{HORIZON_XEM}.csv', index=False)
print(f"Đã lưu {OUTPUT_DIR}/outlier_theo_site_h{HORIZON_XEM}.csv")


--- SAI SO TAI DIEM OUTLIER SO VOI DIEM BINH THUONG, THEO SITE ---
tuong_quan_tai_outlier cao nghia la mo hinh bam theo ca phoc outlier.


,site_id,so_outlier,ty_le_outlier_%,mae_tai_outlier,mae_tai_binh_thuong,tuong_quan_tai_outlier,thuc_te_max_outlier,du_bao_max_outlier
0,2,53,0.442,3.6255,0.5933,0.44246,17.719,18.115
1,1,51,0.425,3.6818,0.6795,0.66621,21.375,19.906
2,42,44,0.371,1.4540,0.1878,0.48552,5.578,5.906
3,15,43,0.359,3.9239,0.7005,0.33386,16.977,18.091
4,23,42,0.352,2.5710,0.4770,0.58955,12.016,11.902
5,22,42,0.352,2.5771,0.5573,0.57603,14.000,13.995
6,20,42,0.352,2.8103,0.6220,0.69241,15.750,16.148
7,6,41,0.341,5.7440,1.0555,0.60622,28.719,26.059
8,14,40,0.335,2.5227,0.5277,0.72915,13.227,13.559
9,26,40,0.336,1.3445,0.2995,0.66781,7.523,7.635


Đã lưu ../../data/model/v4/09_kiem_chung_tre_pha/outlier_theo_site_h1.csv


## 11. Zoom vào các site có nhiều outlier nhất

Vẽ cửa sổ quanh cụm outlier lớn nhất của từng site trong danh sách `SITE_SOI`.
Mặc định lấy 2 site đứng đầu bảng trên; anh đổi tay thành `[19, 24]` nếu muốn soi đúng 2 site đó.


In [22]:
SITE_SOI = df_ol['site_id'].head(2).tolist()
SO_GIO_QUANH = 24   # so gio ve quanh cum outlier

for s in SITE_SOI:
    d = df[df[SITE_COL] == s].sort_values(TIMESTAMP_COL).copy()
    d_ol = d[d['la_outlier']]
    if len(d_ol) == 0:
        print(f"Site {s} khong co outlier, bo qua.")
        continue

    # chon moc outlier co sai lech thuc te lon nhat
    t_tam = d_ol.loc[d_ol['thuc_te'].idxmax(), TIMESTAMP_COL]
    d_ve = d[d[TIMESTAMP_COL].between(t_tam - pd.Timedelta(hours=SO_GIO_QUANH),
                                      t_tam + pd.Timedelta(hours=SO_GIO_QUANH))]
    o_ve = d_ve[d_ve['la_outlier']]

    f = go.Figure()
    f.add_trace(go.Scatter(x=d_ve[TIMESTAMP_COL], y=d_ve['thuc_te'], name='Thực tế',
                           mode='lines+markers', line=dict(width=2.4, color='#1f77b4')))
    f.add_trace(go.Scatter(x=d_ve[TIMESTAMP_COL], y=d_ve['du_bao'], name='Dự báo',
                           mode='lines+markers', line=dict(width=2.0, color='#d62728')))
    f.add_trace(go.Scatter(x=o_ve[TIMESTAMP_COL], y=o_ve['thuc_te'], name='Điểm outlier',
                           mode='markers', marker=dict(size=12, symbol='x', color='#ff7f0e')))
    f.update_layout(title=f'Site {s} - cụm outlier quanh {t_tam} - h{HORIZON_XEM}',
                    xaxis_title='Thời gian', yaxis_title='Sản lượng (kWh)',
                    hovermode='x unified', height=480, template='plotly_white')
    f.show()

    _r = df_ol[df_ol['site_id'] == s].iloc[0]
    print(f"Site {s}: {int(_r['so_outlier'])} diem outlier, "
          f"thuc te max {_r['thuc_te_max_outlier']}, du bao max {_r['du_bao_max_outlier']}, "
          f"tuong quan tai outlier {_r['tuong_quan_tai_outlier']}")
    print("Neu duong do vot len theo dau X thi mo hinh dang hoc ca outlier.")
    print()


Site 2: 53 diem outlier, thuc te max 17.719, du bao max 18.115, tuong quan tai outlier 0.44246
Neu duong do vot len theo dau X thi mo hinh dang hoc ca outlier.



Site 1: 51 diem outlier, thuc te max 21.375, du bao max 19.906, tuong quan tai outlier 0.66621
Neu duong do vot len theo dau X thi mo hinh dang hoc ca outlier.



## 12. Tổng kết

Đọc lại các con số LOCAL (không phải RMSE tổng hợp):

1. Mục 5 - bảng `df_lech`: lệch đỉnh (phút) của TỪNG (site, ngày). Đây là số liệu gốc, đáng tin nhất.
2. Mục 6 - `df_site`: tổng hợp lệch đỉnh theo từng site, site nào lệch trung vị dương lớn là site trễ nặng.
3. Mục 9 - lệch đỉnh tập trung vào khung giờ nào trong ngày.

Không được kết luận "hết trễ" chỉ vì lệch trung vị toàn bộ gần 0 - phải rà bảng `df_lech` xem còn
case nào lệch > 15 phút ở bất kỳ site/ngày nào không. Nếu còn, đó vẫn là bug local cần fix, dù số
liệu tổng hợp có đẹp đến đâu.


In [23]:
print("--- TONG KET KIEM CHUNG LECH DINH (LOCAL, KHONG RMSE) ---")
print(f"Horizon xem                      : h{HORIZON_XEM}")
print(f"So site x ngay da kiem tra        : {df['site_id'].nunique()} site x {len(df_lech):,} ngay")
print(f"Lech dinh trung vi TOAN BO        : {df_lech['lech_phut'].median():+.1f} phut")
so_dich_phai = int((df_lech['lech_phut'] > 0).sum())
print(f"So ngay du bao dich PHAI (tre)    : {so_dich_phai:,}/{len(df_lech):,} ({so_dich_phai/len(df_lech)*100:.1f}%)")
print(f"So site co lech_trung_vi > 0      : {(df_site['lech_trung_vi'] > 0).sum()}/{len(df_site)}")
print(f"Gio dinh lech nang nhat           : {_gio_te} gio")
print(f"Ca lech nang nhat toan bo         : site {SITE_TE_NHAT}, ngay {ngay_te}, lech {_ca_te_nhat['lech_phut']:+.0f} phut")
print()
print("Neu con case lech dinh > 15 phut o bat ky site/ngay nao trong bang df_lech thi van con bug local,")
print("du lech_trung_vi toan bo co gan 0 di nua - khong duoc ket luan 'het tre' chi vi con so trung binh dep.")
print()
print("Cac file da xuat:")
for _f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"   {_f}")


--- TONG KET KIEM CHUNG LECH DINH (LOCAL, KHONG RMSE) ---
Horizon xem                      : h1
So site x ngay da kiem tra        : 40 site x 5,080 ngay
Lech dinh trung vi TOAN BO        : +15.0 phut
So ngay du bao dich PHAI (tre)    : 2,791/5,080 (54.9%)
So site co lech_trung_vi > 0      : 32/40
Gio dinh lech nang nhat           : 16 gio
Ca lech nang nhat toan bo         : site 2, ngay 2021-12-19, lech -495 phut

Neu con case lech dinh > 15 phut o bat ky site/ngay nao trong bang df_lech thi van con bug local,
du lech_trung_vi toan bo co gan 0 di nua - khong duoc ket luan 'het tre' chi vi con so trung binh dep.

Cac file da xuat:
   lech_dinh_moi_ngay_h1.csv
   lech_dinh_theo_gio_h1.csv
   lech_dinh_theo_site_h1.csv
   lech_theo_vi_tri_trong_gio_h1.csv
   outlier_theo_site_h1.csv
